In [1]:
# We are going to create and train a self_supervised PatchTST model, and then fine-tune it using supervised learning.
# The model is based on the paper "Self-Supervised Learning of Patch Transformers for Time Series Classification" by Xu et al. (2022).

# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader, Subset

import random
import os
import numpy as np
import pandas as pd
import math
import sys
from collections import Counter
from itertools import chain
from typing import List
import textwrap

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from google.colab import drive

random.seed(0)
torch.manual_seed(0)

In [2]:
drive.mount('/content/gdrive')

base_dir = "/content/gdrive/MyDrive/CS4782-Final/"
sys.path.append(base_dir)

Mounted at /content/gdrive


In [3]:
%load_ext autoreload
%autoreload 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

Device set to cuda


In [4]:
from PatchTST_self_supervised import PatchTSTSelfSupervised, self_supervised_loss
from PatchTST import PatchTST
from utils import train, val, train_self_supervised, val_self_supervised

In [5]:
from data_loader import TimeSeriesDataset, SelfSupervisedTimeSeriesDataset
from sklearn.preprocessing import StandardScaler

input_length    = 336   # e.g. past 336 steps
forecast_horizon= 96    # e.g. next 96 steps
batch_size      = 32

df   = pd.read_csv(base_dir + "data/ETTh1.csv", parse_dates=["date"])
vars = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
data = df[vars].values

# 2) Determine time‐based split point
n_train_time = int(0.8 * len(data))

# 3) Slice train & val (include overlap of input_length for first val window)
train_raw = data[:n_train_time]
val_raw   = data[n_train_time - input_length :]

# 4) Fit scaler on TRAIN ONLY, then transform both
scaler     = StandardScaler().fit(train_raw)
train_data = scaler.transform(train_raw)
val_data   = scaler.transform(val_raw)

# 5) Build datasets & loaders
selfsup_train_ds = SelfSupervisedTimeSeriesDataset(train_data, input_length)
selfsup_val_ds   = SelfSupervisedTimeSeriesDataset(val_data, input_length)

selfsup_train_loader = DataLoader(selfsup_train_ds, batch_size=32, shuffle=True, drop_last=True)
selfsup_val_loader   = DataLoader(selfsup_val_ds, batch_size=32, shuffle=False)


In [7]:
criterion = nn.MSELoss()
model = PatchTSTSelfSupervised(
    input_length=input_length,
    patch_len=16,
    stride=16,
    n_heads=4,
    d_model=16
).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
print('here')
train_loss_arr, val_loss_arr = train_self_supervised(
    model, selfsup_train_loader, selfsup_val_loader,
    self_supervised_loss=self_supervised_loss, epochs=20,
    optimizer=optimizer, device=device
)

here
Starting training...
Epoch 1/20


train: 100%|██████████| 425/425 [00:08<00:00, 48.92it/s]


Epoch 1, Train Loss: 0.9474, Val Loss = 0.7344
Epoch 2/20


train: 100%|██████████| 425/425 [00:09<00:00, 42.85it/s]


Epoch 2, Train Loss: 0.7316, Val Loss = 0.5882
Epoch 3/20


...

Epoch 18, Train Loss: 0.5431, Val Loss = 0.4513
Epoch 19/20


train: 100%|██████████| 425/425 [00:06<00:00, 61.47it/s]


Epoch 19, Train Loss: 0.5405, Val Loss = 0.4504
Epoch 20/20


train: 100%|██████████| 425/425 [00:06<00:00, 64.06it/s]


Epoch 20, Train Loss: 0.5359, Val Loss = 0.4481
Training finished.


In [10]:
# Get the state_dict of the model's encoder
patch_embed_state_dict = model.patch_embed.state_dict()
transformer_state_dict = model.transformer.state_dict()

In [11]:
df   = pd.read_csv(base_dir + "data/ETTh1.csv", parse_dates=["date"])
vars = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
data = df[vars].values

# 2) Determine time‐based split point
n_train_time = int(0.8 * len(data))

# 3) Slice train & val (include overlap of input_length for first val window)
train_raw = data[:n_train_time]
val_raw   = data[n_train_time - input_length :]

# 4) Fit scaler on TRAIN ONLY, then transform both
scaler     = StandardScaler().fit(train_raw)
train_data = scaler.transform(train_raw)
val_data   = scaler.transform(val_raw)

# 5) Array‐based dataset
class ArrayTimeSeriesDataset(Dataset):
    def __init__(self, data: np.ndarray, input_length: int, horizon: int):
        """
        data: 2D array [T, n_vars] already scaled
        """
        self.data = data
        self.L, self.h = input_length, horizon
        self.n_samples = len(data) - input_length - horizon + 1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.L]                       # [L, n_vars]
        y = self.data[idx + self.L : idx + self.L + self.h]     # [h, n_vars]
        # transpose to [n_vars, seq_len]
        return (
            torch.from_numpy(x.T).float(),
            torch.from_numpy(y.T).float()
        )

# 6) Build datasets & loaders
train_ds = ArrayTimeSeriesDataset(train_data, input_length, forecast_horizon)
val_ds   = ArrayTimeSeriesDataset(val_data,   input_length, forecast_horizon)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, drop_last=False)

In [12]:
# Create a new PatchTST model for supervised training
supervised_model = PatchTST(
    input_length=input_length,
    patch_len=16,
    stride=16,
    n_heads=4,
    d_model=16,
    forecast_horizon=forecast_horizon
).to(device)
optimizer = optim.Adam(supervised_model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
# Load the encoder state_dict into the new model
supervised_model.transformer.load_state_dict(transformer_state_dict)
supervised_model.patch_embed.load_state_dict(patch_embed_state_dict)

supervised_model.transformer.requires_grad = True
supervised_model.patch_embed.requires_grad = True

fine_tune_train_loss_arr, _, fine_tune_val_loss_arr, _ = train(
    supervised_model, train_loader, val_loader,
    criterion=criterion, epochs=10,
    optimizer=optimizer, device=device
)

Starting training...
Epoch 1/10


train: 100%|██████████| 422/422 [00:06<00:00, 63.95it/s]


Epoch 1: Train Loss = 0.4328, Train MAE = 0.4580 | Val   Loss = 0.4302, Val   MAE = 0.4576
Epoch 2/10


train: 100%|██████████| 422/422 [00:06<00:00, 63.36it/s]


Epoch 2: Train Loss = 0.3783, Train MAE = 0.4269 | Val   Loss = 0.4199, Val   MAE = 0.4506
Epoch 3/10


...

Epoch 8: Train Loss = 0.3381, Train MAE = 0.4082 | Val   Loss = 0.4206, Val   MAE = 0.4523
Epoch 9/10


train: 100%|██████████| 422/422 [00:06<00:00, 63.60it/s]


Epoch 9: Train Loss = 0.3348, Train MAE = 0.4067 | Val   Loss = 0.4247, Val   MAE = 0.4534
Epoch 10/10


train: 100%|██████████| 422/422 [00:06<00:00, 65.50it/s]


Epoch 10: Train Loss = 0.3306, Train MAE = 0.4046 | Val   Loss = 0.4345, Val   MAE = 0.4588
Training finished.


In [ ]:
# Create a new PatchTST model for supervised training
normal_supervised_model = PatchTST(
    input_length=input_length,
    patch_len=16,
    stride=16,
    forecast_horizon=forecast_horizon
).to(device)
optimizer = optim.Adam(normal_supervised_model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)

normal_sup_train_loss_arr, _, normal_sup_val_loss_arr, _ = train(
    normal_supervised_model, train_loader, val_loader,
    criterion=criterion, epochs=10,
    optimizer=optimizer, device=device
)

Starting training...
Epoch 1/10


train: 100%|██████████| 422/422 [00:13<00:00, 30.49it/s]


Epoch 1: Train Loss = 0.4189, Train MAE = 0.4490 | Val   Loss = 0.4374, Val   MAE = 0.4635
Epoch 2/10


train: 100%|██████████| 422/422 [00:13<00:00, 30.43it/s]


Epoch 2: Train Loss = 0.3624, Train MAE = 0.4178 | Val   Loss = 0.4358, Val   MAE = 0.4578
Epoch 3/10


...

Epoch 8: Train Loss = 0.2730, Train MAE = 0.3723 | Val   Loss = 0.4509, Val   MAE = 0.4692
Epoch 9/10


train: 100%|██████████| 422/422 [00:13<00:00, 30.21it/s]


Epoch 9: Train Loss = 0.2602, Train MAE = 0.3650 | Val   Loss = 0.4741, Val   MAE = 0.4800
Epoch 10/10


train: 100%|██████████| 422/422 [00:14<00:00, 29.13it/s]


Epoch 10: Train Loss = 0.2460, Train MAE = 0.3568 | Val   Loss = 0.4667, Val   MAE = 0.4795
Training finished.


## Forecast Window Testing

In [ ]:
input_length    = 512   # e.g. past 336 steps
batch_size      = 32

df   = pd.read_csv(base_dir + "data/ETTh1.csv", parse_dates=["date"])
vars = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
data = df[vars].values

# 2) Determine time‐based split point
n_train_time = int(0.8 * len(data))

# 3) Slice train & val (include overlap of input_length for first val window)
train_raw = data[:n_train_time]
val_raw   = data[n_train_time - input_length :]

# 4) Fit scaler on TRAIN ONLY, then transform both
scaler     = StandardScaler().fit(train_raw)
train_data = scaler.transform(train_raw)
val_data   = scaler.transform(val_raw)

# 5) Build datasets & loaders
selfsup_train_ds = SelfSupervisedTimeSeriesDataset(train_data, input_length)
selfsup_val_ds   = SelfSupervisedTimeSeriesDataset(val_data, input_length)

selfsup_train_loader = DataLoader(selfsup_train_ds, batch_size=32, shuffle=True, drop_last=True)
selfsup_val_loader   = DataLoader(selfsup_val_ds, batch_size=32, shuffle=False)

df   = pd.read_csv(base_dir + "data/ETTh1.csv", parse_dates=["date"])
vars = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
data = df[vars].values

# 2) Determine time‐based split point
n_train_time = int(0.8 * len(data))

# 3) Slice train & val (include overlap of input_length for first val window)
train_raw = data[:n_train_time]
val_raw   = data[n_train_time - input_length :]

# 4) Fit scaler on TRAIN ONLY, then transform both
scaler     = StandardScaler().fit(train_raw)
train_data = scaler.transform(train_raw)
val_data   = scaler.transform(val_raw)

# 5) Array‐based dataset
class ArrayTimeSeriesDataset(Dataset):
    def __init__(self, data: np.ndarray, input_length: int, horizon: int):
        """
        data: 2D array [T, n_vars] already scaled
        """
        self.data = data
        self.L, self.h = input_length, horizon
        self.n_samples = len(data) - input_length - horizon + 1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.L]                       # [L, n_vars]
        y = self.data[idx + self.L : idx + self.L + self.h]     # [h, n_vars]
        # transpose to [n_vars, seq_len]
        return (
            torch.from_numpy(x.T).float(),
            torch.from_numpy(y.T).float()
        )


forecast_horizon_windows = [24, 48, 96, 192, 336, 720]

end_val_loss = []

for horizon in forecast_horizon_windows:
    print(f"Horizon: {horizon}")
    print("-" * 20)
    model = PatchTSTSelfSupervised(
        input_length=input_length,
        patch_len=12,
        stride=12,
        n_heads=4,
        d_model=16
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
    train_loss_arr, val_loss_arr = train_self_supervised(
        model, selfsup_train_loader, selfsup_val_loader,
        self_supervised_loss=self_supervised_loss, epochs=20,
        optimizer=optimizer, device=device
    )

    # Get the state_dict of the model's encoder
    transformer_state_dict = model.transformer.state_dict()

    supervised_model = PatchTST(
        input_length=input_length,
        patch_len=12,
        stride=12,
        n_heads=4,
        d_model=16,
        forecast_horizon=horizon
    ).to(device)
    optimizer = optim.Adam(supervised_model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
    # Load the encoder state_dict into the new model
    supervised_model.transformer.load_state_dict(transformer_state_dict)

    supervised_model.transformer.requires_grad = True

    train_ds = ArrayTimeSeriesDataset(train_data, input_length, horizon)
    val_ds   = ArrayTimeSeriesDataset(val_data,   input_length, horizon)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, drop_last=False)

    fine_tune_train_loss_arr, _, fine_tune_val_loss_arr, _ = train(
        supervised_model, train_loader, val_loader,
        criterion=criterion, epochs=10,
        optimizer=optimizer, device=device
    )
    end_val_loss.append(fine_tune_val_loss_arr[-1])

Horizon: 24
--------------------
Starting training...
Epoch 1/20


train: 100%|██████████| 419/419 [00:11<00:00, 37.49it/s]


Epoch 1, Train Loss: 0.9562, Val Loss = 0.8263
Epoch 2/20


train: 100%|██████████| 419/419 [00:11<00:00, 37.36it/s]


Epoch 2, Train Loss: 0.7564, Val Loss = 0.6263
Epoch 3/20


...

Epoch 8: Train Loss = 0.4993, Train MAE = 0.5094 | Val   Loss = 0.7928, Val   MAE = 0.6672
Epoch 9/10


train: 100%|██████████| 397/397 [00:10<00:00, 37.61it/s]


Epoch 9: Train Loss = 0.4879, Train MAE = 0.5045 | Val   Loss = 0.7970, Val   MAE = 0.6751
Epoch 10/10


train: 100%|██████████| 397/397 [00:10<00:00, 37.75it/s]


Epoch 10: Train Loss = 0.4759, Train MAE = 0.4993 | Val   Loss = 0.7865, Val   MAE = 0.6644
Training finished.


In [ ]:
print(list(zip(forecast_horizon_windows, end_val_loss)))

[(24, 0.37278141901580564), (48, 0.40083555766829737), (96, 0.4492974140734043), (192, 0.520216232509289), (336, 0.5842955883103188), (720, 0.7864659209361021)]


## Electricity Stuff

In [ ]:
elec_df = pd.read_csv(base_dir + "data/electricity.txt", index_col=False, header=None)
elec_df.head()

,0,1,2,3,4,5,6,7,8,9,...,311,312,313,314,315,316,317,318,319,320
0,14.0,69.0,234.0,415.0,215.0,1056.0,29.0,840.0,226.0,265.0,...,676.0,372.0,80100.0,4719.0,5002.0,48.0,38.0,1558.0,182.0,2162.0
1,18.0,92.0,312.0,556.0,292.0,1363.0,29.0,1102.0,271.0,340.0,...,805.0,452.0,95200.0,4643.0,6617.0,65.0,47.0,2177.0,253.0,2835.0
2,21.0,96.0,312.0,560.0,272.0,1240.0,29.0,1025.0,270.0,300.0,...,817.0,430.0,96600.0,4285.0,6571.0,64.0,43.0,2193.0,218.0,2764.0
3,20.0,92.0,312.0,443.0,213.0,845.0,24.0,833.0,179.0,211.0,...,801.0,291.0,94500.0,4222.0,6365.0,65.0,39.0,1315.0,195.0,2735.0
4,22.0,91.0,312.0,346.0,190.0,647.0,16.0,733.0,186.0,179.0,...,807.0,279.0,91300.0,4116.0,6298.0,75.0,40.0,1378.0,191.0,2721.0


In [ ]:
# find ten columns with highest variance
small_elec_df = elec_df[elec_df.var().sort_values(ascending=False)[:10].index]

In [ ]:
input_length    = 336   # e.g. past 336 steps
forecast_horizon= 96    # e.g. next 96 steps
batch_size      = 32

data = small_elec_df.values

# 2) Determine time‐based split point
n_train_time = int(0.8 * len(data))

# 3) Slice train & val (include overlap of input_length for first val window)
train_raw = data[:n_train_time]
val_raw   = data[n_train_time - input_length :]

# 4) Fit scaler on TRAIN ONLY, then transform both
scaler     = StandardScaler().fit(train_raw)
train_data = scaler.transform(train_raw)
val_data   = scaler.transform(val_raw)

# 5) Build datasets & loaders
selfsup_train_ds = SelfSupervisedTimeSeriesDataset(train_data, input_length)
selfsup_val_ds   = SelfSupervisedTimeSeriesDataset(val_data, input_length)

selfsup_train_loader = DataLoader(selfsup_train_ds, batch_size=32, shuffle=True, drop_last=True)
selfsup_val_loader   = DataLoader(selfsup_val_ds, batch_size=32, shuffle=False)


In [ ]:
criterion = nn.MSELoss()
model = PatchTSTSelfSupervised(
    input_length=input_length,
    patch_len=16,
    stride=16,
    n_heads=4,
    d_model=16
).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
print('here')
train_loss_arr, val_loss_arr = train_self_supervised(
    model, selfsup_train_loader, selfsup_val_loader,
    self_supervised_loss=self_supervised_loss, epochs=20,
    optimizer=optimizer, device=device
)

# Get the state_dict of the model's encoder
transformer_state_dict = model.transformer.state_dict()

here
Starting training...
Epoch 1/20


train: 100%|██████████| 647/647 [00:14<00:00, 45.43it/s]


Epoch 1, Train Loss: 0.4427, Val Loss = 0.1265
Epoch 2/20


train: 100%|██████████| 647/647 [00:16<00:00, 40.40it/s]


Epoch 2, Train Loss: 0.1333, Val Loss = 0.1080
Epoch 3/20


...

Epoch 18, Train Loss: 0.0768, Val Loss = 0.0718
Epoch 19/20


train: 100%|██████████| 647/647 [00:14<00:00, 46.12it/s]


Epoch 19, Train Loss: 0.0754, Val Loss = 0.0708
Epoch 20/20


train: 100%|██████████| 647/647 [00:14<00:00, 46.06it/s]


Epoch 20, Train Loss: 0.0742, Val Loss = 0.0699
Training finished.


In [ ]:
df   = pd.read_csv(base_dir + "data/ETTh1.csv", parse_dates=["date"])
vars = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
data = df[vars].values

# 2) Determine time‐based split point
n_train_time = int(0.8 * len(data))

# 3) Slice train & val (include overlap of input_length for first val window)
train_raw = data[:n_train_time]
val_raw   = data[n_train_time - input_length :]

# 4) Fit scaler on TRAIN ONLY, then transform both
scaler     = StandardScaler().fit(train_raw)
train_data = scaler.transform(train_raw)
val_data   = scaler.transform(val_raw)

# 5) Array‐based dataset
class ArrayTimeSeriesDataset(Dataset):
    def __init__(self, data: np.ndarray, input_length: int, horizon: int):
        """
        data: 2D array [T, n_vars] already scaled
        """
        self.data = data
        self.L, self.h = input_length, horizon
        self.n_samples = len(data) - input_length - horizon + 1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.L]                       # [L, n_vars]
        y = self.data[idx + self.L : idx + self.L + self.h]     # [h, n_vars]
        # transpose to [n_vars, seq_len]
        return (
            torch.from_numpy(x.T).float(),
            torch.from_numpy(y.T).float()
        )

# 6) Build datasets & loaders
train_ds = ArrayTimeSeriesDataset(train_data, input_length, forecast_horizon)
val_ds   = ArrayTimeSeriesDataset(val_data,   input_length, forecast_horizon)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, drop_last=False)

In [ ]:
# Create a new PatchTST model for supervised training
supervised_model = PatchTST(
    input_length=input_length,
    patch_len=16,
    stride=16,
    n_heads=4,
    d_model=16,
    forecast_horizon=forecast_horizon
).to(device)
optimizer = optim.Adam(supervised_model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
# Load the encoder state_dict into the new model
supervised_model.transformer.load_state_dict(transformer_state_dict)

supervised_model.transformer.requires_grad = True

fine_tune_train_loss_arr, _, fine_tune_val_loss_arr, _ = train(
    supervised_model, train_loader, val_loader,
    criterion=criterion, epochs=10,
    optimizer=optimizer, device=device
)

Starting training...
Epoch 1/10


train: 100%|██████████| 422/422 [00:07<00:00, 60.04it/s]


Epoch 1: Train Loss = 0.4460, Train MAE = 0.4655 | Val   Loss = 0.4397, Val   MAE = 0.4632
Epoch 2/10


train: 100%|██████████| 422/422 [00:06<00:00, 66.59it/s]


Epoch 2: Train Loss = 0.3880, Train MAE = 0.4328 | Val   Loss = 0.4276, Val   MAE = 0.4576
Epoch 3/10


...

Epoch 8: Train Loss = 0.3427, Train MAE = 0.4110 | Val   Loss = 0.4237, Val   MAE = 0.4530
Epoch 9/10


train: 100%|██████████| 422/422 [00:07<00:00, 55.43it/s]


Epoch 9: Train Loss = 0.3393, Train MAE = 0.4091 | Val   Loss = 0.4301, Val   MAE = 0.4559
Epoch 10/10


train: 100%|██████████| 422/422 [00:06<00:00, 64.51it/s]


Epoch 10: Train Loss = 0.3367, Train MAE = 0.4078 | Val   Loss = 0.4207, Val   MAE = 0.4514
Training finished.


In [ ]:
forecast_horizon_windows = [96, 192, 336, 720]

end_val_loss = []

for horizon in forecast_horizon_windows:
    print(f"Horizon: {horizon}")
    print("-" * 20)
    train_ds = ArrayTimeSeriesDataset(train_data, input_length, horizon)
    val_ds   = ArrayTimeSeriesDataset(val_data,   input_length, horizon)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, drop_last=False)

    # Create a new PatchTST model for supervised training
    supervised_model = PatchTST(
        input_length=input_length,
        patch_len=16,
        stride=16,
        n_heads=4,
        d_model=16,
        forecast_horizon=horizon
    ).to(device)
    optimizer = optim.Adam(supervised_model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
    # Load the encoder state_dict into the new model
    supervised_model.transformer.load_state_dict(transformer_state_dict)

    supervised_model.transformer.requires_grad = True

    fine_tune_train_loss_arr, _, fine_tune_val_loss_arr, _ = train(
        supervised_model, train_loader, val_loader,
        criterion=criterion, epochs=5,
        optimizer=optimizer, device=device
    )

    end_val_loss.append(fine_tune_val_loss_arr[-1])

Horizon: 96
--------------------


NameError: name 'ArrayTimeSeriesDataset' is not defined

In [ ]:
print(list(zip(forecast_horizon_windows, end_val_loss)))

[(96, 0.43221974246344474), (192, 0.49039810519773985), (336, 0.5720023172672348), (720, 0.7393650977775968)]
